# Cuaderno ML 2: Predicción del crecimiento de jugadores con Random Forest

**Institución Universitaria Pascual Bravo**
**Asignatura:** Gestión de Datos Masivos
**Docente:** Genry Gutiérrez
**Proyecto:** Steam Game Intelligence
**Infraestructura:** Databricks Lakehouse, Unity Catalog, Delta Lake y Spark MLlib
**Fuente:** capa Gold (`game_growth`, `dim_game`, `twitch_steam_relationship`)

---

## Pregunta de negocio

> **¿Podemos predecir si un juego crecerá al menos un 10 % en jugadores el mes siguiente? ¿La exposición en Twitch mejora esa predicción?**

La correlación descriptiva entre Twitch y el crecimiento en Steam fue casi nula (−0,0032), pero una correlación lineal no detecta relaciones no lineales ni interacciones. Aquí se replantea como un problema de predicción: se entrena el mismo modelo **con y sin** las variables de Twitch y se compara su capacidad predictiva.

## Objetivos

1. Construir desde Gold una tabla juego-mes con variables conocidas en el mes *t* y el objetivo en *t+1*.
2. Evitar la fuga de información con variables solo del pasado y una partición temporal.
3. Entrenar un **Random Forest** con pesos por clase para el desbalance.
4. Evaluar con métricas adecuadas a una clase minoritaria frente a un modelo de referencia.
5. Interpretar la importancia de las variables y el aporte de Twitch.

---
## 1. Aprendizaje supervisado

Se dispone de ejemplos etiquetados $(\mathbf{x}_i, y_i)$:

- $\mathbf{x}_i$: variables conocidas del juego en el mes *t* (jugadores, tendencia, calendario, catálogo, Twitch).
- $y_i \in \{0, 1\}$: **1** si el juego creció al menos un 10 % en *t+1*, **0** en otro caso.

El objetivo es aprender una función que acierte en datos **futuros** que el modelo no vio. Es una **clasificación binaria**.

## 2. Del árbol de decisión al Random Forest

Un **árbol de decisión** divide los datos con preguntas sucesivas, por ejemplo: *¿el juego está a menos del 30 % de su pico? → ¿el mes siguiente hay rebajas de Steam?* Cada división busca grupos lo más puros posible según la **impureza de Gini**:

$$I_G = 1 - \sum_{c} p_c^2$$

Un solo árbol tiende a **sobreajustarse**. El **Random Forest** entrena muchos árboles distintos y combina sus votos. Cada árbol ve una muestra aleatoria de filas y, en cada división, solo un subconjunto aleatorio de variables, así que se equivocan de forma diferente y el promedio es más estable.

### Por qué Random Forest para estos datos

- Los datos son **tabulares** y con escalas muy distintas; los árboles no necesitan estandarización.
- Las relaciones esperadas **no son lineales**: el efecto de las rebajas depende de la etapa del juego, y estar en el top 10 de Twitch no es lo mismo que estar en el puesto 190.
- Entrega la **importancia de cada variable**, lo que permite ver cuánto pesa Twitch.

---
## 3. Desbalance de clases y fuga de información

### Desbalance

Crecer un 10 % en un mes es poco frecuente. Un modelo que siempre responda "no crece" tendría una exactitud alta y no serviría para nada. Por eso:

- La métrica principal es **PR-AUC**, que se concentra en la clase positiva. Un modelo al azar obtiene un PR-AUC igual a la tasa de positivos.
- Se reportan **precisión, recall y F1 de la clase positiva**.
- Se entrena con **pesos por clase**, de modo que las dos clases aportan lo mismo al entrenamiento.

### Fuga de información

Un modelo que "ve el futuro" da métricas excelentes en el notebook y falla en la realidad. Se evita con tres reglas:

1. **Todas las variables usan datos hasta el mes *t*.** Por ejemplo, la distancia al pico usa el máximo histórico *hasta t*.
2. **No se usan** reseñas ni propietarios de `dim_game`: son una foto actual que refleja el éxito futuro del juego.
3. **Partición temporal:** se entrena con 2016–2021 y se evalúa con 2022–2024.

---
## 4. Configuración

In [ ]:
%pip install plotly==5.24.1 --quiet

In [ ]:
import math

import numpy as np
import pandas as pd
import plotly.express as px
from pyspark.ml import Pipeline
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.functions import vector_to_array
from pyspark.sql import Window
from pyspark.sql import functions as F

GOLD = "steam_game_intelligence.gold"

MIN_PLAYERS = 20
THRESHOLD = 0.10
WINDOW_START = "2016-01-01"
WINDOW_END = "2024-08-01"
TRAIN_END = "2021-12-01"
NUM_TREES = 100
MAX_DEPTH = 8
TOP_GENRES = 8
SEED = 42

---
## 5. Carga de la capa Gold

In [ ]:
game_growth = spark.table(f"{GOLD}.game_growth")
dim_game = spark.table(f"{GOLD}.dim_game")
twitch = spark.table(f"{GOLD}.twitch_steam_relationship")

for name, df in [("game_growth", game_growth), ("dim_game", dim_game), ("twitch_steam_relationship", twitch)]:
    print(f"{name}: {df.count():,} filas | {df.select('app_id').distinct().count():,} juegos")

---
## 6. Variables y objetivo

Gold no trae el valor del mes siguiente, así que se calcula con ventanas de Spark (`lag`, `lead` y máximo acumulado) sobre la historia de cada juego. Solo después se recorta a la ventana de análisis, para que los primeros meses tengan su historia previa.

**Objetivo:** `target = 1` si `avg_players(t+1) >= 1,10 × avg_players(t)` y *t+1* es el mes siguiente a *t*. Las filas sin un mes siguiente consecutivo se descartan.

| Grupo | Variable | Cálculo (siempre con datos hasta *t*) |
|---|---|---|
| Historia reciente | `log_avg_players` | `log1p(avg_players)` |
| | `log_change_1m`, `log_change_3m` | Cambio en logaritmo respecto a 1 y 3 meses antes (0 si no hay dato) |
| | `ratio_to_ma3` | `avg_players` ÷ media móvil de 3 meses |
| | `drawdown_from_peak` | `avg_players` ÷ máximo histórico **hasta t** |
| | `peak_to_avg_ratio` | Pico ÷ promedio dentro del mes |
| Calendario y antigüedad | `months_since_release` | Meses desde el lanzamiento (−1 si no hay fecha) |
| | `next_month` | Mes calendario de *t+1* (1–12) |
| | `next_month_is_steam_sale` | 1 si *t+1* es junio, julio o diciembre (rebajas de Steam) |
| Catálogo | `is_free`, `log_price` | Precio en `dim_game` |
| | `genre_*` | Indicadores de los géneros más frecuentes |
| Twitch | `in_twitch_top200` | 1 si el juego está en el top 200 en *t* |
| | `log_hours_watched` | `log1p(horas vistas en t)` |
| | `twitch_rank_filled` | Puesto en el ranking en *t*, 201 si no está |

Las variables `has_*` indican cuándo un valor fue rellenado, para que el modelo distinga un dato real de uno faltante.

In [ ]:
w = Window.partitionBy("app_id").orderBy("month_start")

panel = (
    game_growth
    .filter(F.col("avg_players").isNotNull())
    .select(
        "app_id",
        "name",
        "month_start",
        F.col("avg_players").cast("double").alias("avg_players"),
        F.col("avg_players_ma3").cast("double").alias("avg_players_ma3"),
        F.col("peak_to_avg_ratio").cast("double").alias("peak_to_avg_raw"),
        F.col("months_since_release").cast("double").alias("months_since_release_raw"),
    )
    .withColumn("prev_month", F.lag("month_start").over(w))
    .withColumn("prev_players", F.lag("avg_players").over(w))
    .withColumn("lag3_month", F.lag("month_start", 3).over(w))
    .withColumn("lag3_players", F.lag("avg_players", 3).over(w))
    .withColumn("next_month_start", F.lead("month_start").over(w))
    .withColumn("next_players", F.lead("avg_players").over(w))
    .withColumn("hist_max_players", F.max("avg_players").over(w.rowsBetween(Window.unboundedPreceding, 0)))
)

has_prev = F.coalesce(F.col("prev_month") == F.add_months("month_start", -1), F.lit(False))
has_lag3 = F.coalesce(F.col("lag3_month") == F.add_months("month_start", -3), F.lit(False))
has_next = F.coalesce(F.col("next_month_start") == F.add_months("month_start", 1), F.lit(False))

features = (
    panel.filter(has_next)
    .withColumn("growth_next", F.expr("try_divide(next_players, avg_players)") - 1)
    .withColumn("target", (F.col("next_players") >= (1 + THRESHOLD) * F.col("avg_players")).cast("double"))
    .withColumn("log_avg_players", F.log1p("avg_players"))
    .withColumn("has_prev_month", has_prev.cast("double"))
    .withColumn("log_change_1m", F.when(has_prev, F.log1p("avg_players") - F.log1p("prev_players")).otherwise(0.0))
    .withColumn("has_lag_3m", has_lag3.cast("double"))
    .withColumn("log_change_3m", F.when(has_lag3, F.log1p("avg_players") - F.log1p("lag3_players")).otherwise(0.0))
    .withColumn("ratio_to_ma3", F.coalesce(F.expr("try_divide(avg_players, avg_players_ma3)"), F.lit(1.0)))
    .withColumn("drawdown_from_peak", F.coalesce(F.expr("try_divide(avg_players, hist_max_players)"), F.lit(1.0)))
    .withColumn("peak_to_avg_ratio", F.coalesce("peak_to_avg_raw", F.lit(1.0)))
    .withColumn("has_release_date", F.col("months_since_release_raw").isNotNull().cast("double"))
    .withColumn("months_since_release", F.coalesce("months_since_release_raw", F.lit(-1.0)))
    .withColumn("next_month", F.month(F.add_months("month_start", 1)).cast("double"))
    .withColumn("next_month_is_steam_sale", F.col("next_month").isin(6.0, 7.0, 12.0).cast("double"))
    # Referencia: "crece si creció este mes"
    .withColumn("baseline_pred", (has_prev & (F.col("avg_players") >= (1 + THRESHOLD) * F.col("prev_players"))).cast("double"))
)

catalog_info = dim_game.select("app_id", F.col("price").cast("double").alias("price"), "genres", F.lit(1.0).alias("has_catalog_info"))
twitch_month = twitch.select(
    "app_id",
    "month_start",
    F.col("twitch_rank").cast("double").alias("twitch_rank"),
    F.col("hours_watched").cast("double").alias("hours_watched"),
)

dataset = (
    features
    .join(catalog_info, "app_id", "left")
        .withColumn("is_free", (F.col("price") == 0).cast("double"))
    .withColumn("log_price", F.log1p(F.coalesce("price", F.lit(0.0))))
    .join(twitch_month, ["app_id", "month_start"], "left")
    .withColumn("in_twitch_top200", F.col("twitch_rank").isNotNull().cast("double"))
    .withColumn("log_hours_watched", F.log1p(F.coalesce("hours_watched", F.lit(0.0))))
    .withColumn("twitch_rank_filled", F.coalesce("twitch_rank", F.lit(201.0)))
    .fillna({"is_free": 0.0, "has_catalog_info": 0.0})
    .filter(F.col("month_start").between(WINDOW_START, WINDOW_END))
    .filter(F.col("avg_players") >= MIN_PLAYERS)
    .cache()
)
print(f"Filas juego-mes: {dataset.count():,} | juegos: {dataset.select('app_id').distinct().count():,}")

### 6.1 ¿Qué tan frecuente es crecer un 10 %?

Un umbral muy bajo convierte en "crecimiento" el ruido normal; uno muy alto deja muy pocos positivos para aprender. La línea roja marca el 10 % elegido.

In [ ]:
growth_hist = (
    dataset
    .withColumn("bin", F.round(F.floor(F.greatest(F.lit(-0.5), F.least(F.lit(1.0), "growth_next")) / 0.05) * 0.05, 2))
    .groupBy("bin").count().orderBy("bin").toPandas()
)

fig = px.bar(
    growth_hist,
    x="bin",
    y="count",
    title="Distribución del crecimiento mensual t → t+1 (recortado a [−50 %, +100 %])",
    labels={"bin": "Crecimiento relativo", "count": "Filas juego-mes"},
    template="plotly_white",
)
fig.add_vline(x=THRESHOLD, line_dash="dash", line_color="red", annotation_text=f"umbral {THRESHOLD:.0%}")
fig.update_layout(height=440, xaxis_tickformat=".0%")
fig.show()

print(f"Filas que crecen al menos un {THRESHOLD:.0%}: {dataset.agg(F.avg('target')).first()[0]:.1%}")

### 6.2 Géneros y lista final de variables

Los géneros se eligen solo con los juegos del **periodo de entrenamiento**, para que ninguna decisión use información del periodo de prueba.

In [ ]:
train_games = dataset.filter(F.col("month_start") <= TRAIN_END).select("app_id", "genres").distinct()
top_genres = [
    r["genre"]
    for r in train_games.select(F.explode("genres").alias("genre")).groupBy("genre").count().orderBy(F.desc("count"), "genre").limit(TOP_GENRES).collect()
]

GENRE_COLUMNS = {genre: "genre_" + genre.lower().replace(" ", "_") for genre in top_genres}
for genre, column in GENRE_COLUMNS.items():
    dataset = dataset.withColumn(column, F.coalesce(F.array_contains("genres", genre).cast("double"), F.lit(0.0)))

FEATURES_BASE = [
    "log_avg_players", "log_change_1m", "has_prev_month", "log_change_3m", "has_lag_3m",
    "ratio_to_ma3", "drawdown_from_peak", "peak_to_avg_ratio",
    "months_since_release", "has_release_date", "next_month", "next_month_is_steam_sale",
    "is_free", "log_price", "has_catalog_info",
    *GENRE_COLUMNS.values(),
]
FEATURES_TWITCH = ["in_twitch_top200", "log_hours_watched", "twitch_rank_filled"]
FEATURES_ALL = FEATURES_BASE + FEATURES_TWITCH

print(f"Géneros: {top_genres}")
print(f"Variables sin Twitch: {len(FEATURES_BASE)} | con Twitch: {len(FEATURES_ALL)}")

---
## 7. Partición temporal y pesos por clase

| Conjunto | Meses *t* | Uso |
|---|---|---|
| Entrenamiento | 2016-01 a 2021-12 | Aprender |
| Prueba | 2022-01 a 2024-08 | Evaluar en el "futuro" |

El peso de cada clase se calcula con entrenamiento: $w_1 = \frac{0{,}5}{p}$ y $w_0 = \frac{0{,}5}{1-p}$, donde $p$ es la tasa de positivos. Así las dos clases suman el mismo peso total.

In [ ]:
train_df = dataset.filter(F.col("month_start") <= TRAIN_END)
test_df = dataset.filter(F.col("month_start") > TRAIN_END).cache()

p_train = train_df.agg(F.avg("target")).first()[0]
p_test = test_df.agg(F.avg("target")).first()[0]
weight_pos, weight_neg = 0.5 / p_train, 0.5 / (1 - p_train)
train_df = train_df.withColumn("class_weight", F.when(F.col("target") == 1, weight_pos).otherwise(weight_neg)).cache()

balance = pd.DataFrame([
    {"conjunto": "entrenamiento", "filas": train_df.count(), "pct_positivos": round(100 * p_train, 2)},
    {"conjunto": "prueba", "filas": test_df.count(), "pct_positivos": round(100 * p_test, 2)},
])
display(balance)
print(f"Pesos de clase: positivo={weight_pos:.2f} | negativo={weight_neg:.2f}")

fig = px.bar(
    balance,
    x="conjunto",
    y="pct_positivos",
    text="pct_positivos",
    title=f"% de filas que crecen al menos un {THRESHOLD:.0%}",
    labels={"pct_positivos": "% positivos", "conjunto": ""},
    template="plotly_white",
)
fig.update_layout(height=400)
fig.show()

---
## 8. Entrenamiento

El pipeline tiene dos etapas: `VectorAssembler`, que junta las variables en un vector, y `RandomForestClassifier`, que usa `class_weight` como peso de cada fila. No hace falta `OneHotEncoder` porque los géneros ya son indicadores 0/1, ni `StandardScaler` porque los árboles no dependen de la escala.

Se comparan tres modelos sobre **las mismas filas de prueba**:

| Modelo | Qué es | Para qué sirve |
|---|---|---|
| Referencia | "Crece si creció este mes" | Piso mínimo que el modelo debe superar |
| RF sin Twitch | Historia, calendario y catálogo | Modelo base |
| RF con Twitch | Lo mismo + 3 variables de Twitch | Responde la pregunta de negocio |

In [ ]:
def evaluate(df, name, score_col, pred_col):
    def auc(metric):
        return BinaryClassificationEvaluator(labelCol="target", rawPredictionCol=score_col, metricName=metric).evaluate(df)

    counts = df.groupBy("target", pred_col).count().toPandas()

    def n(real, pred):
        return int(counts.loc[(counts["target"] == real) & (counts[pred_col] == pred), "count"].sum())

    tp, fp, fn, tn = n(1, 1), n(0, 1), n(1, 0), n(0, 0)
    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    return {"modelo": name, "pr_auc": auc("areaUnderPR"), "roc_auc": auc("areaUnderROC"),
            "precision": precision, "recall": recall, "f1": f1, "tp": tp, "fp": fp, "fn": fn, "tn": tn}


def train_forest(feature_cols):
    forest = RandomForestClassifier(
        labelCol="target",
        featuresCol="features",
        weightCol="class_weight",
        numTrees=NUM_TREES,
        maxDepth=MAX_DEPTH,
        subsamplingRate=0.8,
        featureSubsetStrategy="sqrt",
        seed=SEED,
    )
    return Pipeline(stages=[VectorAssembler(inputCols=feature_cols, outputCol="features"), forest]).fit(train_df)


baseline = test_df.withColumn("baseline_score", F.col("log_change_1m"))
results = [evaluate(baseline, "referencia", "baseline_score", "baseline_pred")]

experiments = {"rf_sin_twitch": FEATURES_BASE, "rf_con_twitch": FEATURES_ALL}
models, predictions = {}, {}
for name, cols in experiments.items():
    print(f"Entrenando {name} con {len(cols)} variables...")
    models[name] = train_forest(cols)
    predictions[name] = models[name].transform(test_df).withColumn("probability_growth", vector_to_array("probability")[1]).cache()
    results.append(evaluate(predictions[name], name, "probability", "prediction"))

---
## 9. Evaluación en el periodo de prueba

In [ ]:
results_pd = pd.DataFrame(results)
display(results_pd.round(4))

pr = results_pd.set_index("modelo")["pr_auc"]
print(f"PR-AUC al azar (tasa de positivos en prueba): {p_test:.4f}")
print(f"Diferencia de PR-AUC con Twitch vs. sin Twitch: {pr['rf_con_twitch'] - pr['rf_sin_twitch']:+.4f}")

metrics_long = results_pd.melt(id_vars="modelo", value_vars=["pr_auc", "roc_auc", "precision", "recall", "f1"], var_name="metrica", value_name="valor")
fig = px.bar(
    metrics_long,
    x="metrica",
    y="valor",
    color="modelo",
    barmode="group",
    text_auto=".3f",
    title="Comparación de modelos en prueba",
    labels={"metrica": "", "valor": "Valor", "modelo": "Modelo"},
    template="plotly_white",
)
fig.add_hline(y=p_test, line_dash="dot", line_color="gray", annotation_text="PR-AUC al azar")
fig.update_layout(height=480)
fig.show()

### 9.1 Curvas precisión-recall y matriz de confusión

La curva precisión-recall muestra el compromiso del modelo: si se baja el umbral de decisión se detectan más juegos que crecen (más recall), a costa de más falsas alarmas (menos precisión).

In [ ]:
def pr_curve(df, score_col, name):
    pdf = df.select("target", score_col).toPandas().sort_values(score_col, ascending=False)
    tp = pdf["target"].cumsum().to_numpy()
    precision = tp / np.arange(1, len(pdf) + 1)
    recall = tp / pdf["target"].sum()
    step = max(len(pdf) // 400, 1)
    return pd.DataFrame({"recall": recall[::step], "precision": precision[::step], "modelo": name})


curves = [pr_curve(baseline, "baseline_score", "referencia")]
curves += [pr_curve(scored, "probability_growth", name) for name, scored in predictions.items()]

fig = px.line(
    pd.concat(curves),
    x="recall",
    y="precision",
    color="modelo",
    title="Curvas precisión-recall en prueba",
    labels={"recall": "Recall (juegos que crecen detectados)", "precision": "Precisión (alertas correctas)", "modelo": "Modelo"},
    template="plotly_white",
)
fig.add_hline(y=p_test, line_dash="dot", line_color="gray", annotation_text="azar")
fig.update_layout(height=500, yaxis_range=[0, 1])
fig.show()

In [ ]:
best = results_pd[results_pd["modelo"] != "referencia"].sort_values("pr_auc").iloc[-1]
confusion = pd.DataFrame(
    [[best["tn"], best["fp"]], [best["fn"], best["tp"]]],
    index=["Real: no crece", "Real: crece"],
    columns=["Predicho: no crece", "Predicho: crece"],
)

fig = px.imshow(confusion, text_auto=",", color_continuous_scale="Blues", title=f"Matriz de confusión en prueba ({best['modelo']})", labels={"color": "Filas"})
fig.update_layout(template="plotly_white", height=420)
fig.show()

### 9.2 ¿Twitch ayuda donde hay exposición?

En la mayoría de las filas el juego no está en el top 200 de Twitch, así que sus variables de Twitch valen cero y no pueden cambiar la predicción. Para ver su efecto real se repite la evaluación **solo con las filas en las que el juego sí estaba en el top 200** ese mes.

In [ ]:
in_top200 = F.col("in_twitch_top200") == 1
subset = [evaluate(baseline.filter(in_top200), "referencia", "baseline_score", "baseline_pred")]
subset += [evaluate(scored.filter(in_top200), name, "probability", "prediction") for name, scored in predictions.items()]
display(pd.DataFrame(subset).round(4))

---
## 10. ¿Qué variables usa el modelo?

La importancia de Gini mide cuánto ayudó cada variable a separar bien los casos, sumado en todos los árboles. Es relativa (suma 1) y favorece a las variables continuas con muchos valores, por eso se complementa con la comparación con y sin Twitch, que mide el aporte real en prueba.

In [ ]:
importances = pd.DataFrame({
    "variable": FEATURES_ALL,
    "importancia": models["rf_con_twitch"].stages[-1].featureImportances.toArray(),
}).sort_values("importancia", ascending=False)
importances["es_twitch"] = importances["variable"].isin(FEATURES_TWITCH)

print(f"Importancia total de las variables de Twitch: {importances.loc[importances['es_twitch'], 'importancia'].sum():.2%}")

fig = px.bar(
    importances.head(15),
    x="importancia",
    y="variable",
    color="es_twitch",
    orientation="h",
    title="Top 15 variables del Random Forest con Twitch (importancia de Gini)",
    labels={"importancia": "Importancia relativa", "variable": "", "es_twitch": "Variable de Twitch"},
    color_discrete_map={True: "#9146FF", False: "#1b9e77"},
    template="plotly_white",
)
fig.update_layout(yaxis=dict(autorange="reversed"), height=520)
fig.show()

---
## 11. Escenarios simulados

Se pasan por el modelo con Twitch cinco juegos inventados con perfiles distintos, para explicar cómo reacciona el modelo.

In [ ]:
def scenario(description, genre=None, **values):
    row = {column: 0.0 for column in FEATURES_ALL}
    row.update({
        "has_prev_month": 1.0, "has_lag_3m": 1.0, "has_release_date": 1.0, "has_catalog_info": 1.0,
        "ratio_to_ma3": 1.0, "drawdown_from_peak": 1.0, "peak_to_avg_ratio": 1.3,
        "twitch_rank_filled": 201.0, "log_price": math.log1p(19.99), "next_month": 3.0,
    })
    if genre in GENRE_COLUMNS:
        row[GENRE_COLUMNS[genre]] = 1.0
    row.update(values)
    row["next_month_is_steam_sale"] = float(row["next_month"] in (6.0, 7.0, 12.0))
    row["escenario"] = description
    return row


scenarios = [
    scenario("Juego mediano en caída, antes de las rebajas de verano", "Action",
             log_avg_players=math.log1p(3000), log_change_1m=math.log(0.85), log_change_3m=math.log(0.65),
             ratio_to_ma3=0.88, drawdown_from_peak=0.25, months_since_release=30.0, next_month=6.0),
    scenario("Juego popular en el top 10 de Twitch", "Action",
             log_avg_players=math.log1p(40000), log_change_1m=math.log(1.05), log_change_3m=math.log(1.10),
             ratio_to_ma3=1.03, drawdown_from_peak=0.9, months_since_release=18.0,
             in_twitch_top200=1.0, log_hours_watched=math.log1p(8_000_000), twitch_rank_filled=8.0),
    scenario("Lanzamiento reciente (segundo mes)", "Indie",
             log_avg_players=math.log1p(5000), log_change_1m=math.log(0.74), has_lag_3m=0.0,
             drawdown_from_peak=0.74, months_since_release=1.0, next_month=4.0),
    scenario("Juego gratuito estable y longevo", "Free To Play",
             log_avg_players=math.log1p(15000), log_change_1m=math.log(1.01),
             drawdown_from_peak=0.6, months_since_release=96.0, next_month=10.0, is_free=1.0, log_price=0.0),
    scenario("Juego pequeño con repunte reciente", "Indie",
             log_avg_players=math.log1p(60), log_change_1m=math.log(1.4), log_change_3m=math.log(1.8),
             ratio_to_ma3=1.25, drawdown_from_peak=0.5, months_since_release=48.0, next_month=11.0),
]

scenario_results = (
    models["rf_con_twitch"].transform(spark.createDataFrame(pd.DataFrame(scenarios)))
    .select(
        "escenario",
        F.round(vector_to_array("probability")[1] * 100, 1).alias("probabilidad_pct"),
        F.when(F.col("prediction") == 1, "CRECE").otherwise("NO CRECE").alias("clasificacion"),
    )
    .toPandas()
    .sort_values("probabilidad_pct")
)
display(scenario_results)

fig = px.bar(
    scenario_results,
    x="probabilidad_pct",
    y="escenario",
    color="clasificacion",
    orientation="h",
    text="probabilidad_pct",
    title="Probabilidad de crecer al menos un 10 % el mes siguiente",
    labels={"probabilidad_pct": "Probabilidad de crecimiento (%)", "escenario": "", "clasificacion": "Clasificación"},
    template="plotly_white",
)
fig.add_vline(x=50, line_dash="dot", line_color="gray", annotation_text="umbral de decisión 0,5")
fig.update_layout(height=460, xaxis_range=[0, 100])
fig.show()

---
## 12. Interpretación de resultados

### Desempeño frente a la referencia
- El **RF sin Twitch** obtiene un PR-AUC de **0,43**, el doble que el azar (0,21). La regla de referencia queda por debajo del azar (0,18).
- **Precisión del 38 %:** de cada 100 alertas de "va a crecer", unas 38 aciertan, frente a 21 del azar.
- **Recall del 58 %:** encuentra casi 6 de cada 10 juegos que realmente crecen.
- Que la referencia sea peor que el azar indica **reversión a la media**: si un juego creció este mes, es menos probable que crezca el siguiente.

### ¿Twitch mejora la predicción?
- En toda la prueba, añadir Twitch **no mejora**: el PR-AUC pasa de 0,429 a 0,427.
- Solo con juegos en el top 200 la mejora es mínima (+0,006 de PR-AUC, 3 aciertos más de 213 crecimientos).
- Las variables de Twitch suman apenas un **0,5 %** de la importancia y ninguna entra en el top 15.
- Conclusión: estar en Twitch **no ayuda a predecir** el crecimiento del mes siguiente. Es una conclusión predictiva, no causal.

### Variables que más explican el crecimiento
- **Calendario (42 %):** el mes siguiente y si hay rebajas de Steam.
- **Historia reciente (42 %):** cambio en 3 meses, picos dentro del mes y distancia a la media y al máximo.
- **Catálogo y antigüedad (14 %):** precio, si es gratuito, meses desde el lanzamiento y géneros.

### Aplicaciones en el proyecto
1. **Alerta temprana:** priorizar cada mes los juegos con mayor probabilidad de crecer.
2. **Evaluación de Twitch:** muestra que la presencia en el ranking no agrega información a la que ya dan la historia del juego y el calendario.

### Limitaciones
- El cruce con Twitch solo usa nombres con coincidencia única.
- Faltan variables que explican picos: actualizaciones, contenido nuevo y promociones propias de cada juego.
- El precio de `dim_game` es el actual, no el de cada mes.
- Con otro umbral de decisión distinto de 0,5 cambiaría el equilibrio entre precisión y recall.